In [25]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_community.utilities import SQLDatabase
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from pprint import pprint


config = {'configurable': {'thread_id': '1'}}


In [26]:
# carregar o arquivo SQLite (substitui `db` se já existir)
db = SQLDatabase.from_uri('sqlite:///../../data/contos.sqlite')

print(f'Dialect: {db.dialect}')
print('Available tables:', db.get_usable_table_names())

# Mostrar amostra da primeira tabela, se existir
tables = db.get_usable_table_names()
if tables:
    try:
        print(f'Sample rows from {tables[0]}:')
        print(db.run(f'SELECT * FROM {tables[0]} LIMIT 5;'))
    except Exception as e:
        print('Não foi possível obter amostra:', e)
else:
    print('Nenhuma tabela disponível no banco.')

Dialect: sqlite
Available tables: ['tales']
Sample rows from tales:
[(1, 'Androcles', 'Aesop', 'https://sites.pitt.edu/~dash/type0156.html', 'But shortly afterwards both Androcles and the lion were captured, and the slave was sentenced to be thrown to the lion, after the latter had been kept without food for several days.The emperor and all his court came to see the spectacle, and Androcles was led out into the middle of the arena....'), (2, 'The Slave and the Lion', 'Aesop', 'https://sites.pitt.edu/~dash/type0156.html', "A slave ran away from his master, by whom he had been most cruelly\ntreated, and, in order to avoid capture, betook himself into the desert.\nAs he wandered about in search of food and shelter, he came to a cave,\nwhich he entered and found to by unoccupied. Really, however, it was a\nlion's den,..."), (3, 'Androcles and the Lion', 'Joseph Jacobs', 'https://sites.pitt.edu/~dash/type0156.html', 'It happened in the old days at Rome that a slave named Androcles\nescaped 

In [27]:
sample=eval(db.run(f'SELECT * FROM {tables[0]} LIMIT 5;'))
pprint(sample[0])

(1,
 'Androcles',
 'Aesop',
 'https://sites.pitt.edu/~dash/type0156.html',
 'But shortly afterwards both Androcles and the lion were captured, and the '
 'slave was sentenced to be thrown to the lion, after the latter had been kept '
 'without food for several days.The emperor and all his court came to see the '
 'spectacle, and Androcles was led out into the middle of the arena....')


In [28]:
import sqlite3

# Conectar diretamente ao banco de dados para obter os dados completos
db_path = '../../data/contos.sqlite'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

table_name = tables[0]
cursor.execute(f'SELECT * FROM {table_name} LIMIT 1;')

# Obter nomes das colunas
column_names = [description[0] for description in cursor.description]

# Buscar os dados e formatar como uma lista de dicionários
sample = []
for row in cursor.fetchall():
    sample.append(dict(zip(column_names, row)))

conn.close()

pprint(sample[0])

{'id': 1,
 'origem': 'Aesop',
 'texto_completo': 'But shortly afterwards both Androcles and the lion were '
                   'captured, and the slave was sentenced to be thrown to the '
                   'lion, after the latter had been kept without food for '
                   'several days.The emperor and all his court came to see the '
                   'spectacle, and Androcles was led out into the middle of '
                   'the arena. Soon the lion was let loose from his den, and '
                   'rushed bounding and roaring towards his victim. But as '
                   'soon as he came near to Androcles he recognized his '
                   'friend, and fawned upon him, and licked his hands like a '
                   'friendly dog.The emperor, surprised at this, summoned '
                   'Androcles to him, who told him the whole story. Whereupon '
                   'the slave was pardoned and freed, and the lion let loose '
                   'to his native

In [29]:
from langchain.chat_models import init_chat_model
from langchain_community.agent_toolkits import SQLDatabaseToolkit

model = init_chat_model('gpt-4.1')


toolkit = SQLDatabaseToolkit(db=db, llm=model)

tools = toolkit.get_tools()

for tool in tools:
    print(f'{tool.name}: {tool.description}\n')

sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



In [35]:
from langchain.tools import tool
from langchain_community.utilities import SQLDatabase


@tool
def buscar_historias(titulo: str = None, origem: str = None) -> list:
    """Busca histórias por título ou autor."""
    if origem:
        query = f"SELECT * FROM contos WHERE origem LIKE '%{origem}%'"
    elif titulo:
        query = f"SELECT * FROM contos WHERE autor = '{titulo}'"
    else:
        query = "SELECT * FROM contos LIMIT 10"
        
    res = db.run(query)
    
    return eval(res)


tools.append(buscar_historias)

In [31]:
system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
""".format(
    dialect=db.dialect,
    top_k=5,
)

In [32]:
from langchain.agents import create_agent

agent = create_agent(
    model,
    tools,
    system_prompt=system_prompt,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={'sql_db_query': True},
            description_prefix='Tool execution pending approval',
        ),
    ],
    checkpointer=InMemorySaver(),
    
)

In [33]:
question = "Apresente o texto completo do conto 'The Magic Violin'"
config = {'configurable': {'thread_id': '1'}}

for step in agent.stream(
    {'messages': [{'role': 'user', 'content': question}]},
    config,
    stream_mode='values',
):
    if 'messages' in step:
        step['messages'][-1].pretty_print()
    elif '__interrupt__' in step:
        print('INTERRUPTED:')
        interrupt = step['__interrupt__'][0]
        for request in interrupt.value['action_requests']:
            print(request['description'])
    else:
        pass

================================ Human Message =================================

Apresente o texto completo do conto 'The Magic Violin'
================================== Ai Message ==================================
Tool Calls:
  buscar_historias (call_ntU4OLs54EoV4nHuzvj2Xbgz)
 Call ID: call_ntU4OLs54EoV4nHuzvj2Xbgz
  Args:
    titulo: The Magic Violin


OperationalError: (sqlite3.OperationalError) no such table: contos
[SQL: SELECT * FROM contos WHERE titulo LIKE '%The Magic Violin%']
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [36]:
from langgraph.types import Command

question = "Apresente o texto completo do conto 'The Magic Violin'"
config = {'configurable': {'thread_id': '1'}}

for step in agent.stream(
    Command(resume={'decisions': [{'type': 'approve'}]}),
    config,
    stream_mode='values',
):
    if 'messages' in step:
        step['messages'][-1].pretty_print()
    elif '__interrupt__' in step:
        print('INTERRUPTED:')
        interrupt = step['__interrupt__'][0]
        for request in interrupt.value['action_requests']:
            print(request['description'])
    else:
        pass

================================== Ai Message ==================================
Tool Calls:
  buscar_historias (call_ntU4OLs54EoV4nHuzvj2Xbgz)
 Call ID: call_ntU4OLs54EoV4nHuzvj2Xbgz
  Args:
    titulo: The Magic Violin


OperationalError: (sqlite3.OperationalError) no such table: contos
[SQL: SELECT * FROM contos WHERE titulo LIKE '%The Magic Violin%']
(Background on this error at: https://sqlalche.me/e/20/e3q8)